# Data Exploration for Plant Disease Dataset
This notebook contains the data exploration section extracted from the main training notebook.
Run the cells in order after setting the correct dataset path in the configuration cell.

<h2 style="
text-align:center;
color:#1a237e;
font-weight:bold;
font-size:32px;
text-shadow:2px 2px 8px rgba(26,35,126,0.4);
letter-spacing:1px;">
Imports</h2>

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
import cv2
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from torchvision.utils import make_grid
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import json
import random
from collections import Counter

plt.rcParams["figure.dpi"] = 130
plt.rcParams["font.family"] = "DejaVu Sans"

<h2 style="
text-align:center;
color:#1a237e;
font-weight:bold;
font-size:32px;
text-shadow:2px 2px 8px rgba(26,35,126,0.4);
letter-spacing:1px;">
Configuration</h2>

In [2]:
CONFIG = {
    "data_dir": "./dataset",
    "image_size": 128,
    "batch_size": 128,
    "num_epochs": 5,
    "learning_rate": 1e-4,
    "num_workers": 4,
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "checkpoint_path": "./best_model.pth",
}

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])

print("Device:", CONFIG["device"])
print("CUDA Available:", torch.cuda.is_available())
print("Set CONFIG['data_dir'] to your dataset root containing 'train' and 'valid' folders.")

Device: cpu
CUDA Available: False
Set CONFIG['data_dir'] to your dataset root containing 'train' and 'valid' folders.


<h2 style="
text-align:center;
color:#1a237e;
font-weight:bold;
font-size:32px;
text-shadow:2px 2px 8px rgba(26,35,126,0.4);
letter-spacing:1px;">
Load Dataset Paths</h2>

In [3]:
train_dir = os.path.join(CONFIG["data_dir"], "train")
valid_dir = os.path.join(CONFIG["data_dir"], "valid")

if not os.path.isdir(train_dir) or not os.path.isdir(valid_dir):
    raise FileNotFoundError(
        f"Expected folders not found. Please set CONFIG['data_dir'] correctly. Checked: {train_dir} and {valid_dir}"
    )

classes = sorted(os.listdir(train_dir))
num_classes = len(classes)
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

train_paths, train_labels = [], []
valid_paths, valid_labels = [], []

for cls in classes:
    for img in os.listdir(os.path.join(train_dir, cls)):
        train_paths.append(os.path.join(train_dir, cls, img))
        train_labels.append(class_to_idx[cls])

for cls in classes:
    for img in os.listdir(os.path.join(valid_dir, cls)):
        valid_paths.append(os.path.join(valid_dir, cls, img))
        valid_labels.append(class_to_idx[cls])

print(f"Number of classes  : {num_classes}")
print(f"Training samples   : {len(train_paths)}")
print(f"Validation samples : {len(valid_paths)}")

FileNotFoundError: Expected folders not found. Please set CONFIG['data_dir'] correctly. Checked: ./dataset\train and ./dataset\valid

<h2 style="
text-align:center;
color:#1a237e;
font-weight:bold;
font-size:32px;
text-shadow:2px 2px 8px rgba(26,35,126,0.4);
letter-spacing:1px;">
EDA: Class Distribution</h2>

In [ ]:
train_counts = Counter(train_labels)
class_names_short = [c.replace("___", "\n").replace("_", " ") for c in classes]
counts = [train_counts[i] for i in range(num_classes)]

palette = sns.color_palette("RdYlGn", num_classes)

fig, ax = plt.subplots(figsize=(20, 7))
bars = ax.barh(class_names_short, counts, color=palette, edgecolor="white", linewidth=0.5)
ax.set_xlabel("Number of Images", fontsize=13)
ax.set_title("Training Set - Class Distribution", fontsize=16, fontweight="bold", pad=15)
ax.invert_yaxis()

for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 80, bar.get_y() + bar.get_height() / 2,
            f"{count:,}", va="center", fontsize=9, color="#333333")

ax.spines[["top", "right"]].set_visible(False)
ax.set_facecolor("#f9f9f9")
fig.patch.set_facecolor("#ffffff")
plt.tight_layout()
plt.show()

<h2 style="
text-align:center;
color:#1a237e;
font-weight:bold;
font-size:32px;
text-shadow:2px 2px 8px rgba(26,35,126,0.4);
letter-spacing:1px;">
EDA: Sample Images Per Class</h2>

In [ ]:
fig, axes = plt.subplots(4, 9, figsize=(22, 11))
axes = axes.flatten()

for idx, cls in enumerate(classes[:36]):
    cls_dir = os.path.join(train_dir, cls)
    img_file = random.choice(os.listdir(cls_dir))
    img = Image.open(os.path.join(cls_dir, img_file)).convert("RGB")
    img = img.resize((128, 128))
    axes[idx].imshow(img)
    axes[idx].set_title(cls.split("___")[-1].replace("_", " ")[:18], fontsize=6.5, pad=2)
    axes[idx].axis("off")

for idx in range(len(classes), len(axes)):
    axes[idx].axis("off")

fig.suptitle("Sample Images Across All Disease Classes", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

<h2 style="
text-align:center;
color:#1a237e;
font-weight:bold;
font-size:32px;
text-shadow:2px 2px 8px rgba(26,35,126,0.4);
letter-spacing:1px;">
EDA: Image Size Distribution</h2>

In [ ]:
sample_paths = random.sample(train_paths, min(2000, len(train_paths)))
widths, heights = [], []

for p in sample_paths:
    img = Image.open(p)
    w, h = img.size
    widths.append(w)
    heights.append(h)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(widths, bins=30, color="#2ecc71", edgecolor="white")
axes[0].set_title("Width Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Width (px)")
axes[0].set_ylabel("Count")
axes[0].spines[["top", "right"]].set_visible(False)

axes[1].hist(heights, bins=30, color="#e74c3c", edgecolor="white")
axes[1].set_title("Height Distribution", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Height (px)")
axes[1].set_ylabel("Count")
axes[1].spines[["top", "right"]].set_visible(False)

fig.suptitle("Image Dimensions in Training Set", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

<h2 style="
text-align:center;
color:#1a237e;
font-weight:bold;
font-size:32px;
text-shadow:2px 2px 8px rgba(26,35,126,0.4);
letter-spacing:1px;">
EDA: Average Pixel Intensity Per Plant</h2>

In [ ]:
plants = list(set([c.split("___")[0] for c in classes]))
plant_brightness = {}

for plant in plants:
    plant_classes = [c for c in classes if c.startswith(plant)]
    values = []
    for cls in plant_classes:
        cls_dir = os.path.join(train_dir, cls)
        for img_file in random.sample(os.listdir(cls_dir), min(20, len(os.listdir(cls_dir)))):
            img = Image.open(os.path.join(cls_dir, img_file)).convert("L")
            values.append(np.mean(np.array(img)))
    plant_brightness[plant] = np.mean(values)

sorted_plants = sorted(plant_brightness, key=plant_brightness.get)
vals = [plant_brightness[p] for p in sorted_plants]

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(sorted_plants)))
ax.barh(sorted_plants, vals, color=colors, edgecolor="white")
ax.set_xlabel("Mean Pixel Intensity", fontsize=12)
ax.set_title("Average Image Brightness by Plant Type", fontsize=14, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
ax.set_facecolor("#fafafa")
plt.tight_layout()
plt.show()

<h2 style="
text-align:center;
color:#1a237e;
font-weight:bold;
font-size:32px;
text-shadow:2px 2px 8px rgba(26,35,126,0.4);
letter-spacing:1px;">
EDA: Healthy vs Diseased Ratio</h2>

In [ ]:
healthy_count  = sum(1 for c in classes if "healthy" in c.lower())
diseased_count = num_classes - healthy_count

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    [healthy_count, diseased_count],
    labels=["Healthy", "Diseased"],
    autopct="%1.1f%%",
    colors=["#2ecc71", "#e74c3c"],
    startangle=90,
    wedgeprops=dict(edgecolor="white", linewidth=2),
    textprops=dict(fontsize=13)
)
for autotext in autotexts:
    autotext.set_fontsize(13)
    autotext.set_fontweight("bold")

ax.set_title("Healthy vs Diseased Classes", fontsize=15, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()